# Transformers: Self-Attention from Scratch

This notebook builds the core Transformer mechanism step by step with NumPy:
1. **Token embeddings** and why we need them
2. **Positional encoding** -- injecting sequence order
3. **Scaled dot-product attention** -- Query, Key, Value
4. **Multi-head attention** -- parallel attention heads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. Token Embeddings

Each token (word/subword) is mapped to a dense vector of dimension $d_{\text{model}}$.
We use a simple lookup table (embedding matrix).

In [ ]:
# Toy vocabulary
vocab = {'the': 0, 'cat': 1, 'sat': 2, 'on': 3, 'mat': 4, '<PAD>': 5}
vocab_size = len(vocab)
d_model = 8  # embedding dimension (small for demonstration)

# Random embedding matrix (in practice, learned during training)
E = np.random.randn(vocab_size, d_model) * 0.1

# Encode a sentence
sentence = ['the', 'cat', 'sat', 'on', 'the', 'mat']
token_ids = [vocab[w] for w in sentence]
X = E[token_ids]  # shape: (seq_len, d_model)

print(f'Sentence: {sentence}')
print(f'Token IDs: {token_ids}')
print(f'Embedding matrix shape: {E.shape}')
print(f'Input tensor shape: {X.shape}')

## 2. Positional Encoding

Self-attention is permutation-invariant, so we add positional information:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding."""
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, np.newaxis]
    div_term = 10000 ** (2 * np.arange(d_model // 2) / d_model)
    PE[:, 0::2] = np.sin(positions / div_term)
    PE[:, 1::2] = np.cos(positions / div_term)
    return PE

PE = positional_encoding(len(sentence), d_model)
X_pos = X + PE  # add positional info to embeddings

# Visualise positional encoding for a longer sequence
PE_vis = positional_encoding(50, 32)
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(PE_vis.T, aspect='auto', cmap='RdBu')
ax.set_xlabel('Position')
ax.set_ylabel('Dimension')
ax.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im, ax=ax)
plt.show()

## 3. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q** (Query): what am I looking for?
- **K** (Key): what do I contain?
- **V** (Value): what information do I provide?

In [ ]:
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """Compute scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
    weights = softmax(scores)          # attention weights
    output = weights @ V               # weighted sum of values
    return output, weights

# Project input into Q, K, V using random weight matrices
d_k = d_v = d_model
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

Q = X_pos @ W_Q
K = X_pos @ W_K
V = X_pos @ W_V

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f'Q shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}')
print(f'Output shape: {output.shape}')
print(f'Attention weights shape: {attn_weights.shape}')

# Visualise attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_weights, cmap='Blues')
ax.set_xticks(range(len(sentence)))
ax.set_xticklabels(sentence, rotation=45)
ax.set_yticks(range(len(sentence)))
ax.set_yticklabels(sentence)
ax.set_title('Attention Weights')
ax.set_xlabel('Key')
ax.set_ylabel('Query')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4. Multi-Head Attention

Instead of one attention function, use $h$ parallel heads, each with its own projections:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

In [ ]:
def multi_head_attention(X, n_heads=2):
    """Multi-head attention with random projections."""
    seq_len, d_model = X.shape
    d_k = d_model // n_heads
    
    all_heads = []
    all_weights = []
    for h in range(n_heads):
        W_Q_h = np.random.randn(d_model, d_k) * 0.1
        W_K_h = np.random.randn(d_model, d_k) * 0.1
        W_V_h = np.random.randn(d_model, d_k) * 0.1
        
        Q_h = X @ W_Q_h
        K_h = X @ W_K_h
        V_h = X @ W_V_h
        
        head_out, weights = scaled_dot_product_attention(Q_h, K_h, V_h)
        all_heads.append(head_out)
        all_weights.append(weights)
    
    # Concatenate heads
    concat = np.concatenate(all_heads, axis=-1)  # (seq_len, d_model)
    
    # Output projection
    W_O = np.random.randn(d_model, d_model) * 0.1
    output = concat @ W_O
    
    return output, all_weights

mha_output, head_weights = multi_head_attention(X_pos, n_heads=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, ax in enumerate(axes):
    im = ax.imshow(head_weights[i], cmap='Blues')
    ax.set_xticks(range(len(sentence)))
    ax.set_xticklabels(sentence, rotation=45)
    ax.set_yticks(range(len(sentence)))
    ax.set_yticklabels(sentence)
    ax.set_title(f'Head {i+1}')
    plt.colorbar(im, ax=ax)
plt.suptitle('Multi-Head Attention Weights', y=1.02)
plt.tight_layout()
plt.show()

print('Each head can learn to attend to different linguistic patterns.')

## Key Takeaways

- **Self-attention** allows each token to attend to every other token in parallel.
- **Positional encoding** injects sequence order (sinusoidal or learned).
- The $\sqrt{d_k}$ scaling prevents softmax saturation.
- **Multi-head attention** enables the model to capture diverse relationships.
- A full Transformer block adds **LayerNorm**, **residual connections**, and a **feed-forward network**.

**Next:** Prompt engineering -- how to communicate effectively with Transformers.